In [2]:
import json  # 导入 json 模块。 #

vocab_path = "/root/autodl-tmp/vocab/vocab.json"  # 改成你的 vocab.json 路径。 #

with open(vocab_path, "r", encoding="utf-8") as f:  # 打开 vocab.json。 #
    vocab = json.load(f)  # 读取词表字典。 #

special_tokens = ["<pad>", "[PAD]", "<PAD>", "pad", "<cls>", "[CLS]", "<eos>", "[EOS]", "</s>", "<bos>", "[SEP]"]  # 常见特殊 token。 #

print("===== 词表中可能的特殊 token =====")  # 打印标题。 #
for token in special_tokens:  # 遍历候选特殊 token。 #
    if token in vocab:  # 如果该 token 存在。 #
        print(f"{token}: {vocab[token]}")  # 打印 token 和 id。 #

print("\n===== 词表中包含 pad/cls/eos 字样的 token =====")  # 打印标题。 #
for token, idx in vocab.items():  # 遍历整个词表。 #
    low = token.lower()  # 转小写便于匹配。 #
    if ("pad" in low) or ("cls" in low) or ("eos" in low):  # 匹配相关 token。 #
        print(f"{token}: {idx}")  # 打印结果。 #

===== 词表中可能的特殊 token =====
<pad>: 0
<cls>: 1

===== 词表中包含 pad/cls/eos 字样的 token =====
<pad>: 0
<cls>: 1


In [2]:
import json  # 导入 json 模块。 #
from pathlib import Path  # 导入路径处理模块。 #
from collections import Counter  # 导入计数器模块。 #
import pandas as pd  # 导入 pandas。 #

# ========================= #
# 1) 在这里填写你的文件路径。 #
# ========================= #
vocab_json_path = "/root/autodl-tmp/vocab/vocab.json"  # 第一个 vocab.json 路径。 #
gene_vocab_path = "/root/autodl-tmp/vocab/gene_vocabulary.jsonl"  # 第二个 gene_vocabulary.jsonl 路径。 #

# ===================================================== #
# 2) 读取普通 vocab.json：格式应为 {token: token_id, ...}。 #
# ===================================================== #
def load_vocab_json_as_dict(path):  # 定义读取 vocab.json 的函数。 #
    path = Path(path)  # 转成 Path 对象。 #
    with open(path, "r", encoding="utf-8") as f:  # 打开文件。 #
        obj = json.load(f)  # 读取 json。 #
    if not isinstance(obj, dict):  # 如果不是字典则报错。 #
        raise ValueError(f"{path} 不是标准的 dict 格式 vocab.json。")  # 抛出异常。 #
    vocab_dict = {}  # 初始化词表字典。 #
    for k, v in obj.items():  # 遍历每一项。 #
        vocab_dict[str(k)] = int(v)  # 统一成 str -> int。 #
    return vocab_dict  # 返回结果。 #

# ====================================================================== #
# 3) 读取 gene_vocabulary.jsonl 或 json：提取为 {ensembl_id: token_id, ...}。 #
# ====================================================================== #
def load_gene_vocab_as_dict(path):  # 定义读取 gene_vocabulary 的函数。 #
    path = Path(path)  # 转成 Path 对象。 #
    text = path.read_text(encoding="utf-8").strip()  # 先读取原始文本。 #

    records = None  # 初始化记录变量。 #

    try:  # 先尝试按普通 JSON 读取。 #
        obj = json.loads(text)  # 尝试解析整个文件。 #
        if isinstance(obj, list):  # 如果是列表格式。 #
            records = obj  # 直接作为记录。 #
        elif isinstance(obj, dict):  # 如果是字典格式。 #
            if "data" in obj and isinstance(obj["data"], list):  # 如果 data 字段里是列表。 #
                records = obj["data"]  # 取 data。 #
            else:  # 否则格式不符合预期。 #
                raise ValueError("JSON 是 dict，但未找到可解析的列表记录。")  # 抛出异常。 #
    except Exception:  # 如果整体 JSON 解析失败。 #
        records = []  # 初始化为空列表，准备按 jsonl 逐行读取。 #
        with open(path, "r", encoding="utf-8") as f:  # 打开文件。 #
            for line_no, line in enumerate(f, start=1):  # 逐行遍历。 #
                line = line.strip()  # 去掉首尾空白。 #
                if not line:  # 跳过空行。 #
                    continue  # 进入下一行。 #
                try:  # 尝试解析该行。 #
                    rec = json.loads(line)  # 按 json 解析一行。 #
                    records.append(rec)  # 加入记录列表。 #
                except Exception as e:  # 如果某行解析失败。 #
                    raise ValueError(f"{path} 第 {line_no} 行不是合法 JSON：{e}")  # 抛出异常。 #

    gene_dict = {}  # 初始化 gene -> token_id 字典。 #
    duplicate_gene_rows = []  # 记录重复 gene 的情况。 #
    missing_rows = []  # 记录缺字段的行。 #

    for idx, rec in enumerate(records):  # 遍历所有记录。 #
        if not isinstance(rec, dict):  # 如果某条记录不是字典。 #
            missing_rows.append((idx, "record_not_dict"))  # 记下来。 #
            continue  # 跳过。 #

        gene = rec.get("ensembl_id", None)  # 优先取 ensembl_id。 #
        token_id = rec.get("token_id", None)  # 取 token_id。 #

        if gene is None or token_id is None:  # 如果缺少关键字段。 #
            missing_rows.append((idx, rec))  # 记录问题行。 #
            continue  # 跳过。 #

        gene = str(gene)  # 统一 gene 为字符串。 #
        token_id = int(token_id)  # 统一 token_id 为整数。 #

        if gene in gene_dict and gene_dict[gene] != token_id:  # 如果同一个 gene 对应多个不同 token_id。 #
            duplicate_gene_rows.append((gene, gene_dict[gene], token_id, idx))  # 记录冲突。 #

        gene_dict[gene] = token_id  # 写入字典，后出现的会覆盖前面的。 #

    return gene_dict, duplicate_gene_rows, missing_rows, records  # 返回解析结果。 #

# ========================================================== #
# 4) 比较两个词表的匹配程度：token 重合、id 重合、映射完全一致率。 #
# ========================================================== #
def compare_vocab_dicts(vocab1, vocab2):  # 定义比较函数。 #
    keys1 = set(vocab1.keys())  # 第一个词表的 token 集合。 #
    keys2 = set(vocab2.keys())  # 第二个词表的 token 集合。 #
    ids1 = set(vocab1.values())  # 第一个词表的 id 集合。 #
    ids2 = set(vocab2.values())  # 第二个词表的 id 集合。 #

    common_keys = keys1 & keys2  # 共有 token。 #
    only_in_1 = keys1 - keys2  # 只在 vocab1 中的 token。 #
    only_in_2 = keys2 - keys1  # 只在 vocab2 中的 token。 #

    common_ids = ids1 & ids2  # 共有 id。 #
    only_ids_1 = ids1 - ids2  # 只在 vocab1 中的 id。 #
    only_ids_2 = ids2 - ids1  # 只在 vocab2 中的 id。 #

    exact_match_keys = []  # 记录 token 和 id 都一致的项。 #
    id_mismatch_keys = []  # 记录同 token 但 id 不一致的项。 #

    for k in sorted(common_keys):  # 遍历共有 token。 #
        if vocab1[k] == vocab2[k]:  # 如果 token_id 完全一致。 #
            exact_match_keys.append((k, vocab1[k], vocab2[k]))  # 加入完全一致列表。 #
        else:  # 如果 token_id 不一致。 #
            id_mismatch_keys.append((k, vocab1[k], vocab2[k]))  # 加入不一致列表。 #

    result = {  # 组装结果字典。 #
        "n_vocab1_tokens": len(keys1),  # vocab1 token 数。 #
        "n_vocab2_tokens": len(keys2),  # vocab2 token 数。 #
        "n_common_tokens": len(common_keys),  # 共有 token 数。 #
        "n_only_in_vocab1": len(only_in_1),  # vocab1 独有 token 数。 #
        "n_only_in_vocab2": len(only_in_2),  # vocab2 独有 token 数。 #
        "token_overlap_rate_vs_vocab1": len(common_keys) / len(keys1) if len(keys1) > 0 else 0.0,  # 相对 vocab1 的 token 重合率。 #
        "token_overlap_rate_vs_vocab2": len(common_keys) / len(keys2) if len(keys2) > 0 else 0.0,  # 相对 vocab2 的 token 重合率。 #
        "n_vocab1_ids": len(ids1),  # vocab1 id 数。 #
        "n_vocab2_ids": len(ids2),  # vocab2 id 数。 #
        "n_common_ids": len(common_ids),  # 共有 id 数。 #
        "id_overlap_rate_vs_vocab1": len(common_ids) / len(ids1) if len(ids1) > 0 else 0.0,  # 相对 vocab1 的 id 重合率。 #
        "id_overlap_rate_vs_vocab2": len(common_ids) / len(ids2) if len(ids2) > 0 else 0.0,  # 相对 vocab2 的 id 重合率。 #
        "n_exact_token_id_match": len(exact_match_keys),  # token 与 id 完全一致的数量。 #
        "n_same_token_but_id_mismatch": len(id_mismatch_keys),  # token 相同但 id 不一致的数量。 #
        "exact_match_rate_among_common_tokens": len(exact_match_keys) / len(common_keys) if len(common_keys) > 0 else 0.0,  # 在共有 token 中完全一致率。 #
        "common_keys": common_keys,  # 共有 token 集合。 #
        "only_in_1": only_in_1,  # vocab1 独有 token 集合。 #
        "only_in_2": only_in_2,  # vocab2 独有 token 集合。 #
        "common_ids": common_ids,  # 共有 id 集合。 #
        "only_ids_1": only_ids_1,  # vocab1 独有 id 集合。 #
        "only_ids_2": only_ids_2,  # vocab2 独有 id 集合。 #
        "exact_match_keys": exact_match_keys,  # 完全一致明细。 #
        "id_mismatch_keys": id_mismatch_keys,  # 不一致明细。 #
    }  # 结果字典结束。 #

    return result  # 返回比较结果。 #

# ============================================= #
# 5) 读取两个文件，并输出核心匹配统计。 #
# ============================================= #
vocab1 = load_vocab_json_as_dict(vocab_json_path)  # 读取 vocab.json。 #
vocab2, duplicate_gene_rows, missing_rows, raw_records = load_gene_vocab_as_dict(gene_vocab_path)  # 读取 gene_vocabulary。 #
result = compare_vocab_dicts(vocab1, vocab2)  # 比较两个词表。 #

print("========== 基本统计 ==========")  # 打印标题。 #
print(f"vocab.json token 数: {result['n_vocab1_tokens']}")  # 打印 vocab1 token 数。 #
print(f"gene_vocabulary token 数: {result['n_vocab2_tokens']}")  # 打印 vocab2 token 数。 #
print(f"共有 token 数: {result['n_common_tokens']}")  # 打印共有 token 数。 #
print(f"vocab.json 独有 token 数: {result['n_only_in_vocab1']}")  # 打印 vocab1 独有 token 数。 #
print(f"gene_vocabulary 独有 token 数: {result['n_only_in_vocab2']}")  # 打印 vocab2 独有 token 数。 #
print()  # 空行。 #

print("========== token 名称重合率 ==========")  # 打印标题。 #
print(f"相对 vocab.json 的 token 重合率: {result['token_overlap_rate_vs_vocab1']:.4%}")  # 打印 token 重合率。 #
print(f"相对 gene_vocabulary 的 token 重合率: {result['token_overlap_rate_vs_vocab2']:.4%}")  # 打印 token 重合率。 #
print()  # 空行。 #

print("========== token_id 重合率 ==========")  # 打印标题。 #
print(f"vocab.json 的唯一 id 数: {result['n_vocab1_ids']}")  # 打印 vocab1 唯一 id 数。 #
print(f"gene_vocabulary 的唯一 id 数: {result['n_vocab2_ids']}")  # 打印 vocab2 唯一 id 数。 #
print(f"共有 id 数: {result['n_common_ids']}")  # 打印共有 id 数。 #
print(f"相对 vocab.json 的 id 重合率: {result['id_overlap_rate_vs_vocab1']:.4%}")  # 打印 id 重合率。 #
print(f"相对 gene_vocabulary 的 id 重合率: {result['id_overlap_rate_vs_vocab2']:.4%}")  # 打印 id 重合率。 #
print()  # 空行。 #

print("========== token->id 映射一致性 ==========")  # 打印标题。 #
print(f"共有 token 中，token_id 完全一致的数量: {result['n_exact_token_id_match']}")  # 打印完全一致数。 #
print(f"共有 token 中，token_id 不一致的数量: {result['n_same_token_but_id_mismatch']}")  # 打印不一致数。 #
print(f"共有 token 中的完全一致率: {result['exact_match_rate_among_common_tokens']:.4%}")  # 打印完全一致率。 #
print()  # 空行。 #

print("========== gene_vocabulary 文件质量检查 ==========")  # 打印标题。 #
print(f"缺失关键字段的记录数: {len(missing_rows)}")  # 打印缺字段记录数。 #
print(f"同一 ensembl_id 对应多个 token_id 的冲突数: {len(duplicate_gene_rows)}")  # 打印 gene 冲突数。 #

# ======================================================= #
# 6) 展示一些不一致明细，方便你快速看问题出在哪里。 #
# ======================================================= #
mismatch_df = pd.DataFrame(result["id_mismatch_keys"], columns=["token", "vocab_json_id", "gene_vocab_id"])  # 构造不一致表。 #
only_in_1_df = pd.DataFrame(sorted(list(result["only_in_1"])), columns=["token_only_in_vocab_json"])  # 构造 vocab1 独有 token 表。 #
only_in_2_df = pd.DataFrame(sorted(list(result["only_in_2"])), columns=["token_only_in_gene_vocab"])  # 构造 vocab2 独有 token 表。 #
duplicate_gene_df = pd.DataFrame(duplicate_gene_rows, columns=["ensembl_id", "old_token_id", "new_token_id", "record_index"])  # 构造 gene 冲突表。 #

print()  # 空行。 #
print("========== token 相同但 id 不一致：前 20 条 ==========")  # 打印标题。 #
display(mismatch_df.head(20))  # 展示前 20 条不一致记录。 #

print("========== 只在 vocab.json 中出现的 token：前 20 条 ==========")  # 打印标题。 #
display(only_in_1_df.head(20))  # 展示前 20 条。 #

print("========== 只在 gene_vocabulary 中出现的 token：前 20 条 ==========")  # 打印标题。 #
display(only_in_2_df.head(20))  # 展示前 20 条。 #

print("========== gene_vocabulary 中同一 ensembl_id 的重复冲突：前 20 条 ==========")  # 打印标题。 #
display(duplicate_gene_df.head(20))  # 展示前 20 条。 #

# ====================================================== #
# 7) 额外检查：看看 vocab.json 里的特殊 token（如 <junk*>）数量。 #
# ====================================================== #
special_tokens = [k for k in vocab1.keys() if k.startswith("<") and k.endswith(">")]  # 找出 vocab.json 里的特殊 token。 #
print()  # 空行。 #
print("========== vocab.json 特殊 token 检查 ==========")  # 打印标题。 #
print(f"特殊 token 数量: {len(special_tokens)}")  # 打印特殊 token 数量。 #
print("前 20 个特殊 token：")  # 打印提示。 #
print(special_tokens[:20])  # 打印前 20 个特殊 token。 #

# ======================================================= #
# 8) 保存结果到本地，后续你可以直接看 csv。 #
# ======================================================= #
out_dir = Path("./vocab_compare_results")  # 定义输出目录。 #
out_dir.mkdir(parents=True, exist_ok=True)  # 创建输出目录。 #

pd.DataFrame([  # 构造汇总统计表。 #
    {
        "n_vocab_json_tokens": result["n_vocab1_tokens"],  # vocab1 token 数。 #
        "n_gene_vocab_tokens": result["n_vocab2_tokens"],  # vocab2 token 数。 #
        "n_common_tokens": result["n_common_tokens"],  # 共有 token 数。 #
        "token_overlap_rate_vs_vocab_json": result["token_overlap_rate_vs_vocab1"],  # 相对 vocab1 的 token 重合率。 #
        "token_overlap_rate_vs_gene_vocab": result["token_overlap_rate_vs_vocab2"],  # 相对 vocab2 的 token 重合率。 #
        "n_vocab_json_ids": result["n_vocab1_ids"],  # vocab1 id 数。 #
        "n_gene_vocab_ids": result["n_vocab2_ids"],  # vocab2 id 数。 #
        "n_common_ids": result["n_common_ids"],  # 共有 id 数。 #
        "id_overlap_rate_vs_vocab_json": result["id_overlap_rate_vs_vocab1"],  # 相对 vocab1 的 id 重合率。 #
        "id_overlap_rate_vs_gene_vocab": result["id_overlap_rate_vs_vocab2"],  # 相对 vocab2 的 id 重合率。 #
        "n_exact_token_id_match": result["n_exact_token_id_match"],  # 完全一致数。 #
        "n_same_token_but_id_mismatch": result["n_same_token_but_id_mismatch"],  # token 相同但 id 不一致数。 #
        "exact_match_rate_among_common_tokens": result["exact_match_rate_among_common_tokens"],  # 共有 token 中的完全一致率。 #
        "n_missing_rows_in_gene_vocab": len(missing_rows),  # gene_vocab 缺字段记录数。 #
        "n_duplicate_gene_conflicts_in_gene_vocab": len(duplicate_gene_rows),  # gene_vocab 冲突数。 #
        "n_special_tokens_in_vocab_json": len(special_tokens),  # vocab1 特殊 token 数。 #
    }
]).to_csv(out_dir / "summary.csv", index=False, encoding="utf-8-sig")  # 保存汇总表。 #

mismatch_df.to_csv(out_dir / "token_same_but_id_mismatch.csv", index=False, encoding="utf-8-sig")  # 保存 id 不一致明细。 #
only_in_1_df.to_csv(out_dir / "only_in_vocab_json.csv", index=False, encoding="utf-8-sig")  # 保存 vocab1 独有 token。 #
only_in_2_df.to_csv(out_dir / "only_in_gene_vocab.csv", index=False, encoding="utf-8-sig")  # 保存 vocab2 独有 token。 #
duplicate_gene_df.to_csv(out_dir / "duplicate_gene_conflicts.csv", index=False, encoding="utf-8-sig")  # 保存 gene 冲突明细。 #

print()  # 空行。 #
print(f"结果已保存到: {out_dir.resolve()}")  # 打印输出目录。 #

========== 基本统计 ==========
vocab.json token 数: 62720
gene_vocabulary token 数: 62710
共有 token 数: 62710
vocab.json 独有 token 数: 10
gene_vocabulary 独有 token 数: 0

========== token 名称重合率 ==========
相对 vocab.json 的 token 重合率: 99.9841%
相对 gene_vocabulary 的 token 重合率: 100.0000%

========== token_id 重合率 ==========
vocab.json 的唯一 id 数: 62720
gene_vocabulary 的唯一 id 数: 62710
共有 id 数: 62710
相对 vocab.json 的 id 重合率: 99.9841%
相对 gene_vocabulary 的 id 重合率: 100.0000%

========== token->id 映射一致性 ==========
共有 token 中，token_id 完全一致的数量: 62710
共有 token 中，token_id 不一致的数量: 0
共有 token 中的完全一致率: 100.0000%

========== gene_vocabulary 文件质量检查 ==========
缺失关键字段的记录数: 0
同一 ensembl_id 对应多个 token_id 的冲突数: 0

========== token 相同但 id 不一致：前 20 条 ==========


,token,vocab_json_id,gene_vocab_id


========== 只在 vocab.json 中出现的 token：前 20 条 ==========


,token_only_in_vocab_json
0,<cls>
1,<eoc>
2,<junk0>
3,<junk1>
4,<junk2>
5,<junk3>
6,<junk4>
7,<junk5>
8,<junk6>
9,<pad>


========== 只在 gene_vocabulary 中出现的 token：前 20 条 ==========


,token_only_in_gene_vocab


========== gene_vocabulary 中同一 ensembl_id 的重复冲突：前 20 条 ==========


,ensembl_id,old_token_id,new_token_id,record_index



========== vocab.json 特殊 token 检查 ==========
特殊 token 数量: 10
前 20 个特殊 token：
['<junk6>', '<junk4>', '<junk0>', '<junk1>', '<junk2>', '<eoc>', '<junk3>', '<pad>', '<cls>', '<junk5>']

结果已保存到: /root/autodl-tmp/vocab/vocab_compare_results
